In [18]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [44]:
class Value:
    def __init__(self, data, _children=(),_op='',label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"
    

    def __add__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data + other.data,(self , other), "+")

        def backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = backward

        return out
    
    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data,(self,other), "*")

        def backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = backward
        return out

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):
        return self + other
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __rtruediv__(self, other):
        return other * self **-1
        
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other,(self,), f'**({other})')
        def backward():
            self.grad += other*self.data**(other-1)*out.grad
        out._backward = backward
        return out

    def __truediv__(self,other):
        return self * other ** -1 

    def __neg__(self):
        return -1 * self

    def __sub__(self,other):
        return self + (-other)

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t , (self,), "tanh")
        def backward():
            self.grad += (1 - t**2)*out.grad
        out._backward= backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward
        
    

In [46]:
a = Value(2.0); b = Value(-3.0)
c = a * b + a
c.grad = 1.0
c._backward()
for node in c._prev:
    node._backward()
assert a.grad == -2.0, f"a.grad = {a.grad}"
assert b.grad == 2.0, f"b.grad = {b.grad}"
print("test 1 passed: mul + add gradients")

test 1 passed: mul + add gradients


In [ ]:
a = Value(2.0); b = Value(-3.0)
c = a * b + a
c.grad = 1.0
c._backward()
for node in c._prev:
    node._backward()
c.backward()

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = a * b + a
c.grad = 1.0
c.backward()
assert a.grad == -2.0, f"a.grad = {a.grad}"
assert b.grad == 2.0, f"b.grad = {b.grad}"
print("test 1 passed: add and mul gradients")

AssertionError: a.grad = 1.0